In [1]:
from pathlib import Path
import json
import re
import warnings

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore", category=FutureWarning)

RANDOM_STATE = 661
PROJECT_YEARS = list(range(2018, 2025))
PM25_THRESHOLD = 25.0
MIN_VALID_PM_HOURS = 18
MIN_VALID_COUNT_HOURS = 24
MAX_FIRE_DISTANCE_KM = 500.0
PREFILTER_DISTANCE_KM = 520.0

# Locate the repository project folder and the Source code/Datasets folder.
# This supports the GitHub structure:
#   repository root/
#     Source code/
#       this notebook
#       Datasets/
#         ntepa_greater_darwin_hourly_2018_2024.csv
#         fire_archive_SV-C2_*.csv
#
# It also supports running VS Code/Jupyter from either the repository root
# or directly from the Source code folder.
possible_project_folders = [
    Path.cwd(),
    Path.cwd() / "Source code",
    Path.cwd().parent,
    Path.cwd().parent / "Source code",
]

project_folder = None
data_folder = None

for folder in possible_project_folders:
    folder = folder.resolve()

    # Preferred repository structure: Source code/Datasets
    candidate_data_folder = folder / "Datasets"
    if (candidate_data_folder / "ntepa_greater_darwin_hourly_2018_2024.csv").exists():
        project_folder = folder
        data_folder = candidate_data_folder
        break

    # Backward-compatible fallback: datasets stored directly beside the notebook.
    if (folder / "ntepa_greater_darwin_hourly_2018_2024.csv").exists():
        project_folder = folder
        data_folder = folder
        break

if project_folder is None or data_folder is None:
    raise FileNotFoundError(
        "The NT EPA dataset could not be found. Expected the project structure "
        "'Source code/Datasets/ntepa_greater_darwin_hourly_2018_2024.csv'. "
        "Run this notebook from the repository root or from the Source code folder."
    )

output_root = project_folder / "outputs_prt661"
table_dir = output_root / "tables"
processed_dir = output_root / "processed"

for folder in [output_root, table_dir, processed_dir]:
    folder.mkdir(parents=True, exist_ok=True)

print("Project folder:", project_folder)
print("Dataset folder:", data_folder)
print("Output folder:", output_root)


Project folder: C:\Users\nguye\Documents\PRT661---DATA-SCIENCE-PRACTICE---Dan5---Theme2\Source code
Dataset folder: C:\Users\nguye\Documents\PRT661---DATA-SCIENCE-PRACTICE---Dan5---Theme2\Source code\Datasets
Output folder: C:\Users\nguye\Documents\PRT661---DATA-SCIENCE-PRACTICE---Dan5---Theme2\Source code\outputs_prt661


In [ ]:
ntepa_file = data_folder / "ntepa_greater_darwin_hourly_2018_2024.csv"
df = pd.read_csv(ntepa_file)
df["datetime_local"] = pd.to_datetime(df["datetime_local"], errors="coerce")
df["date"] = df["datetime_local"].dt.normalize()
df["year"] = df["datetime_local"].dt.year

required_columns = {
    "datetime_local", "station", "latitude", "longitude",
    "pm25_ug_m3", "pm10_ug_m3", "relative_humidity_pct",
    "air_temperature_c", "wind_speed_m_s", "wind_direction_deg",
    "air_pressure_hpa", "rainfall_mm",
}
missing_columns = required_columns.difference(df.columns)
if missing_columns:
    raise ValueError(f"NT EPA file is missing required columns: {sorted(missing_columns)}")

quality_columns = [
    "pm25_ug_m3", "pm10_ug_m3", "relative_humidity_pct",
    "air_temperature_c", "wind_speed_m_s", "wind_direction_deg",
    "air_pressure_hpa", "rainfall_mm",
]

overview = pd.DataFrame({
    "metric": [
        "rows", "columns", "date_min", "date_max", "stations",
        "full_row_duplicates", "station_timestamp_duplicates"
    ],
    "value": [
        len(df), df.shape[1], df["datetime_local"].min(), df["datetime_local"].max(),
        ", ".join(sorted(df["station"].dropna().astype(str).unique())),
        int(df.duplicated().sum()),
        int(df.duplicated(["station", "datetime_local"]).sum()),
    ]
})
print("\nRaw NT EPA overview:")
print(overview.to_string(index=False))
overview.to_csv(table_dir / "epa_raw_overview.csv", index=False)

missing_by_year_station = (
    df.groupby(["year", "station"])[quality_columns]
      .agg(lambda s: s.isna().mean() * 100)
      .round(2)
      .reset_index()
)
missing_by_year_station.to_csv(
    table_dir / "epa_missing_pct_year_station_variable.csv", index=False
)

raw_summary = df[quality_columns].describe(
    percentiles=[0.001, 0.01, 0.05, 0.5, 0.95, 0.99, 0.999]
).T
raw_summary.to_csv(table_dir / "epa_raw_numeric_summary.csv")
print("\nRaw numeric summary:")
print(raw_summary.round(3).to_string())


In [ ]:
df_clean = df.copy()
cleaning_log = []

def log_change(variable, rule, before_nonmissing, after_nonmissing):
    cleaning_log.append({
        "variable": variable,
        "rule": rule,
        "values_set_or_changed": int(before_nonmissing - after_nonmissing),
    })

# Remove physically impossible negative PM concentrations.
for col in ["pm25_ug_m3", "pm10_ug_m3"]:
    before = int(df_clean[col].notna().sum())
    df_clean.loc[pd.to_numeric(df_clean[col], errors="coerce") < 0, col] = np.nan
    after = int(df_clean[col].notna().sum())
    log_change(col, "negative concentration -> NaN", before, after)

# Pressure audit. The newer pipeline identified Stokes Hill on an ~1 atmosphere scale.
pressure_profile = (
    df_clean.groupby("station")["air_pressure_hpa"]
    .agg(["count", "median", "min", "max"])
    .reset_index()
)
pressure_profile["conversion_factor"] = np.where(
    pressure_profile["median"].between(0.8, 1.2, inclusive="both"),
    1013.25,
    1.0,
)
pressure_profile.to_csv(table_dir / "epa_pressure_unit_audit.csv", index=False)
print("\nPressure unit audit:")
print(pressure_profile.round(4).to_string(index=False))

for row in pressure_profile.itertuples(index=False):
    if row.conversion_factor != 1.0:
        mask = df_clean["station"].eq(row.station) & df_clean["air_pressure_hpa"].notna()
        df_clean.loc[mask, "air_pressure_hpa"] = (
            df_clean.loc[mask, "air_pressure_hpa"] * row.conversion_factor
        )
        cleaning_log.append({
            "variable": "air_pressure_hpa",
            "rule": f"{row.station}: ~1-atmosphere scale converted to hPa using x1013.25",
            "values_set_or_changed": int(mask.sum()),
        })

plausible_ranges = {
    "relative_humidity_pct": (0.0, 100.0),
    "air_temperature_c": (-20.0, 60.0),
    "wind_speed_m_s": (0.0, 80.0),
    "air_pressure_hpa": (850.0, 1100.0),
}

for col, (low, high) in plausible_ranges.items():
    before = int(df_clean[col].notna().sum())
    valid = pd.to_numeric(df_clean[col], errors="coerce").between(low, high, inclusive="both")
    df_clean.loc[df_clean[col].notna() & ~valid, col] = np.nan
    after = int(df_clean[col].notna().sum())
    log_change(col, f"outside [{low}, {high}] -> NaN", before, after)

# Wind direction is circular and valid in [0, 360).
before = int(df_clean["wind_direction_deg"].notna().sum())
valid_wind_dir = pd.to_numeric(df_clean["wind_direction_deg"], errors="coerce").between(
    0.0, 360.0, inclusive="left"
)
df_clean.loc[df_clean["wind_direction_deg"].notna() & ~valid_wind_dir, "wind_direction_deg"] = np.nan
after = int(df_clean["wind_direction_deg"].notna().sum())
log_change("wind_direction_deg", "outside [0, 360) -> NaN", before, after)

# Rainfall behaviour audit and harmonised hourly rainfall.
rain_profile_rows = []
for station, group in df_clean.sort_values("datetime_local").groupby("station"):
    s = pd.to_numeric(group["rainfall_mm"], errors="coerce")
    nonmissing = s.dropna()
    rain_profile_rows.append({
        "station": station,
        "nonmissing_pct": 100 * s.notna().mean(),
        "median_raw": nonmissing.median() if not nonmissing.empty else np.nan,
        "p95_raw": nonmissing.quantile(0.95) if not nonmissing.empty else np.nan,
        "zero_fraction_raw": nonmissing.eq(0).mean() if not nonmissing.empty else np.nan,
        "n_unique_raw": nonmissing.nunique(),
    })

rain_profile = pd.DataFrame(rain_profile_rows)
rain_profile["cumulative_counter_flag"] = (
    (rain_profile["median_raw"] > 100)
    & (rain_profile["zero_fraction_raw"].fillna(1.0) < 0.20)
)
rain_profile.to_csv(table_dir / "epa_rainfall_station_behaviour_audit.csv", index=False)
print("\nRainfall behaviour audit:")
print(rain_profile.round(3).to_string(index=False))

df_clean = df_clean.sort_values(["station", "datetime_local"]).copy()
df_clean["rainfall_hourly_mm"] = np.nan
cumulative_stations = set(rain_profile.loc[rain_profile["cumulative_counter_flag"], "station"])

for station, idx in df_clean.groupby("station").groups.items():
    g = df_clean.loc[idx].sort_values("datetime_local")
    raw_rain = pd.to_numeric(g["rainfall_mm"], errors="coerce")

    if station in cumulative_stations:
        diffs = raw_rain.diff()
        hour_gap = g["datetime_local"].diff().dt.total_seconds().div(3600)
        derived = diffs.where(hour_gap.eq(1))
        derived = derived.where(derived >= 0)
        derived = derived.where(derived <= 300)
        df_clean.loc[g.index, "rainfall_hourly_mm"] = derived
    else:
        direct = raw_rain.where(raw_rain.between(0, 300))
        df_clean.loc[g.index, "rainfall_hourly_mm"] = direct

cleaning_log_df = pd.DataFrame(cleaning_log)
cleaning_log_df.to_csv(table_dir / "epa_cleaning_log.csv", index=False)
print("\nCleaning log:")
print(cleaning_log_df.to_string(index=False))

cleaned_ntepa_file = project_folder / "NT_EPA_Air_Quality_Cleaned_2018_2024.csv"
df_clean.to_csv(cleaned_ntepa_file, index=False)
print("\nCleaned NT EPA shape:", df_clean.shape)
print("Saved:", cleaned_ntepa_file.name)
